# 03 — Clean data; retain Bronze and Silver

**Learner scaffold · RetailFlow dataset 1.0.0**

Course lessons: Day 1, Lesson 4; Day 2, Lesson 7.

Goal: Repair six rows, quarantine six, drop two duplicates, and retain 60 current sales.

Before running: Run 00. Day 1: complete through 03-04. Day 2: teach persistence in 03-05.

Writes: <prefix>_bronze_initial, <prefix>_quarantine, <prefix>_duplicate_rows, <prefix>_silver_sales. These are snapshots for the exercise.

Follow the playlist module order. Build Buddy will help you create the missing numbered exercise cells. The instructor has a separate completed reference. Step IDs are stable anchors for that authoring work. Saved outputs are blank unless explicitly verified locally; a passed local check does not imply Databricks execution.


## Your notebook grows through Build Buddy

Configuration and instructor utility cells are supplied so you can focus on the learning task. The numbered exercise cells have no solutions. Read the playlist instructions, attempt each cell, run it, explain its output, then ask for a hint if needed. A blank exercise cell is incomplete work. Do not use Run All expecting this scaffold to be a finished pipeline. The course revisits notebooks: Lesson 5 runs 02-02, 02-03 and 02-05; Lesson 6 then runs 02-04.


## Configuration

Use the same assigned names in every notebook. Each notebook defines its own variables and helpers; it does not depend on another notebook’s Python memory. Persistent tables are the handoff between notebooks.


In [0]:
# Edit these four values in EVERY notebook before running it.
CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "retailflow_files"
TABLE_PREFIX = "rf_retailflow"

import re
for name in [CATALOG, SCHEMA, VOLUME, TABLE_PREFIX]:
    assert re.fullmatch(r"[a-z][a-z0-9_]*", name), "Use lowercase letters, digits, and underscores; start with a letter."
assert not any(name.startswith("your_") for name in [CATALOG, SCHEMA, VOLUME]), "Replace the example names with your assigned names."
assert TABLE_PREFIX != "rf_yourname", "Choose the table prefix assigned to you."
DATASET_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/retailflow-v1.0.0"
print("Dataset:", DATASET_ROOT)
print("Table prefix:", f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_")


Dataset: /Volumes/workspace/default/retailflow_files/retailflow-v1.0.0
Table prefix: workspace.default.rf_retailflow_


## Supplied support cells

These utilities keep repeated setup and checks in one place inside this notebook. The Python cleaner deliberately processes this tiny fixture on the driver so beginners can inspect each row. Spark handles DataFrames, joins, summaries, and Delta tables. This is a teaching pattern for 80 sales, not a large-scale ingestion design. The helper definitions below are part of the notebook and can be inspected. Learners will construct selected parts through Build Buddy in the later authoring phase.


In [0]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation
from datetime import date, datetime
from collections import Counter, defaultdict
import csv, json, re

def read_rows(relative_path):
    with (Path(DATASET_ROOT) / relative_path).open(encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))

def money(value):
    return Decimal(value).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

def business_payload(row):
    return {key: value for key, value in row.items() if key != 'source_row_id'}

def business_records(rows):
    return sorted([business_payload(row) for row in rows], key=lambda row: row['sale_id'])

def dimension_maps():
    return (
        {r['store_id']: r for r in read_rows('data/stores.csv')},
        {r['product_id']: r for r in read_rows('data/products.csv')},
        {r['customer_id']: r for r in read_rows('data/customers.csv')},
    )

def clean_sales(raw_rows):
    """Readable Python cleaning for this tiny fixture; no distributed UDF required."""
    stores, products, customers = dimension_maps()
    candidates, rejected, repaired = [], [], []
    for raw in sorted(raw_rows, key=lambda row: row['source_row_id']):
        row = {key: value.strip() for key, value in raw.items()}
        for key in ['store_id', 'product_id', 'customer_id', 'payment_method']:
            row[key] = row[key].upper()
        if row['customer_id'] == 'UNKNOWN':
            row['customer_id'] = ''
        reason = None
        if any(not row[k] for k in ['sale_id', 'source_row_id', 'sale_date', 'store_id', 'product_id']):
            reason = 'REQUIRED_MISSING'
        if not reason:
            try:
                fmt = '%d/%m/%Y' if '/' in row['sale_date'] else '%Y-%m-%d'
                pattern = r'\d{2}/\d{2}/\d{4}' if '/' in row['sale_date'] else r'\d{4}-\d{2}-\d{2}'
                if not re.fullmatch(pattern, row['sale_date']):
                    raise ValueError('Date format mismatch')
                row['sale_date'] = datetime.strptime(row['sale_date'], fmt).date().isoformat()
            except ValueError:
                reason = 'DATE_INVALID'
        if not reason:
            if row['product_id'] not in products:
                reason = 'PRODUCT_UNKNOWN'
            elif row['store_id'] not in stores:
                reason = 'STORE_UNKNOWN'
            elif row['customer_id'] and row['customer_id'] not in customers:
                reason = 'CUSTOMER_UNKNOWN'
        if not reason:
            try:
                if not re.fullmatch(r'\d+', row['quantity']) or int(row['quantity']) < 1:
                    reason = 'QUANTITY_INVALID'
                elif not re.fullmatch(r'\d+', row['record_version']) or int(row['record_version']) < 1:
                    reason = 'VERSION_INVALID'
                else:
                    row['unit_price'] = format(money(row['unit_price'] or products[row['product_id']]['list_price']), '.2f')
                    row['discount_pct'] = row['discount_pct'] or '0'
                    price, discount = Decimal(row['unit_price']), Decimal(row['discount_pct'])
                    if not price.is_finite() or price <= 0:
                        reason = 'PRICE_INVALID'
                    elif not discount.is_finite() or not 0 <= discount <= 100:
                        reason = 'DISCOUNT_INVALID'
                    elif row['payment_method'] not in ['UPI', 'CARD', 'CASH']:
                        reason = 'PAYMENT_INVALID'
            except (ValueError, InvalidOperation):
                reason = 'NUMBER_INVALID'
        if reason:
            rejected.append(dict(raw, rejection_reason=reason))
        else:
            if row != raw:
                repaired.append(row['source_row_id'])
            candidates.append(row)
    groups = defaultdict(list)
    for row in candidates:
        groups[(row['sale_id'], row['record_version'])].append(row)
    accepted, duplicate_ids = [], []
    raw_by_id = {row['source_row_id']: row for row in raw_rows}
    for group in groups.values():
        if any(business_payload(row) != business_payload(group[0]) for row in group):
            rejected.extend(dict(raw_by_id[row['source_row_id']], rejection_reason='VERSION_CONFLICT') for row in group)
        else:
            accepted.append(group[0])
            duplicate_ids.extend(row['source_row_id'] for row in group[1:])
    return accepted, rejected, duplicate_ids, repaired

def current_versions(rows):
    state = {}
    for row in rows:
        old = state.get(row['sale_id'])
        if old is None or int(row['record_version']) > int(old['record_version']):
            state[row['sale_id']] = row
    return sorted(state.values(), key=lambda row: row['sale_id'])

def visible_returns(stage, sales_rows):
    rows = [r for r in read_rows('data/returns.csv') if stage == 'final' or r['available_batch'] == '01']
    sales = {row['sale_id']: row for row in sales_rows}
    assert len({r['return_id'] for r in rows}) == len(rows)
    returned = Counter()
    for row in rows:
        sale = sales[row['sale_id']]
        assert row['available_batch'] in ['01', '02']
        assert date.fromisoformat(row['return_date']) >= date.fromisoformat(sale['sale_date'])
        quantity = int(row['return_quantity'])
        assert quantity > 0
        returned[row['sale_id']] += quantity
        assert returned[row['sale_id']] <= int(sale['quantity'])
        assert Decimal(row['refund_amount']) == money(quantity * Decimal(sale['unit_price']) * (1 - Decimal(sale['discount_pct']) / 100))
    return rows


In [0]:
from pyspark.sql import functions as F, types as T

SALES_COLUMNS = ['source_row_id', 'sale_id', 'sale_date', 'store_id', 'product_id', 'customer_id', 'quantity', 'unit_price', 'discount_pct', 'payment_method', 'record_version']
SALES_SCHEMA = T.StructType([
    T.StructField(name, T.DateType() if name == 'sale_date' else
                  T.IntegerType() if name in ['quantity', 'record_version'] else
                  T.DecimalType(12, 2) if name == 'unit_price' else
                  T.DecimalType(5, 2) if name == 'discount_pct' else T.StringType(), True)
    for name in SALES_COLUMNS
])

def sales_frame(rows):
    typed = []
    for row in rows:
        values = []
        for name in SALES_COLUMNS:
            value = row[name]
            if name == 'sale_date': value = date.fromisoformat(value)
            elif name in ['quantity', 'record_version']: value = int(value)
            elif name in ['unit_price', 'discount_pct']: value = Decimal(value)
            elif name == 'customer_id' and value == '': value = None
            values.append(value)
        typed.append(tuple(values))
    return spark.createDataFrame(typed, SALES_SCHEMA)

def raw_frame(relative_path):
    return spark.read.option('header', True).option('inferSchema', False).option('mode', 'FAILFAST').csv(f'{DATASET_ROOT}/{relative_path}')

def table_name(suffix):
    assert re.fullmatch(r'[a-z][a-z0-9_]*', suffix)
    return f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_{suffix}'

def assert_expected(frame, relative_path, ignore_provenance=False):
    columns = [c for c in frame.columns if not (ignore_provenance and c == 'source_row_id')]
    actual = frame.select(*columns)
    # Read full files before dropping provenance so schema fields match CSV positions.
    expected = spark.read.schema(frame.schema).option('header', True).option('mode', 'FAILFAST').csv(f'{DATASET_ROOT}/checks/expected/{relative_path}').select(*columns)
    assert actual.exceptAll(expected).limit(1).count() == 0, f'Extra/different rows: {relative_path}'
    assert expected.exceptAll(actual).limit(1).count() == 0, f'Missing rows: {relative_path}'
    print('PASS:', relative_path)

def collected_sales(frame):
    rows = []
    for record in frame.select(*SALES_COLUMNS).collect():
        row = record.asDict()
        row['sale_date'] = row['sale_date'].isoformat()
        row['quantity'] = str(row['quantity'])
        row['record_version'] = str(row['record_version'])
        row['unit_price'] = format(row['unit_price'], '.2f')
        row['discount_pct'] = format(row['discount_pct'].normalize(), 'f')
        row['customer_id'] = row['customer_id'] or ''
        rows.append(row)
    return sorted(rows, key=lambda row: row['sale_id'])

def merge_counts(current, incoming):
    joined = incoming.alias('s').join(current.alias('t'), F.col('s.sale_id') == F.col('t.sale_id'), 'left')
    business_columns = [c for c in SALES_COLUMNS if c != 'source_row_id']
    equal = F.lit(True)
    for name in business_columns:
        equal = equal & F.col(f's.{name}').eqNullSafe(F.col(f't.{name}'))
    conflicts = joined.where((F.col('s.record_version') == F.col('t.record_version')) & ~equal)
    assert conflicts.limit(1).count() == 0, 'Equal-version business conflict: investigate before merge.'
    counts = {'insert': 0, 'update': 0, 'noop': 0}
    for row in joined.select(
        F.when(F.col('t.sale_id').isNull(), 'insert')
         .when(F.col('s.record_version') > F.col('t.record_version'), 'update')
         .otherwise('noop').alias('action')
    ).groupBy('action').count().collect():
        counts[row['action']] = row['count']
    return counts

def merge_into_silver(incoming):
    from delta.tables import DeltaTable
    assert incoming.groupBy('sale_id').count().where('count > 1').limit(1).count() == 0
    target = table_name('silver_sales')
    counts = merge_counts(spark.table(target), incoming)
    (DeltaTable.forName(spark, target).alias('t')
        .merge(incoming.alias('s'), 't.sale_id = s.sale_id')
        .whenMatchedUpdateAll(condition='s.record_version > t.record_version')
        .whenNotMatchedInsertAll().execute())
    return counts


## 03-01: Inspect the messy export

**Core concept Vibe:** Raw evidence helps explain why a record was changed or rejected.

**Build Buddy action:** Inspect the deliberate issue rows before cleaning.

**Expected checkpoint:** Whitespace, a slash-formatted date, missing values, UNKNOWN customer, and negative quantity are visible.


In [0]:
# Dataset files sit directly in the volume root, not under a retailflow-v1.0.0 subfolder.
DATASET_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
dirty_rows = read_rows('data/sales_dirty.csv')
assert len(dirty_rows) == 68
for row in dirty_rows:
    if row['source_row_id'] in ['D-001', 'D-003', 'D-004', 'D-005', 'D-006', 'D-063']:
        print(row)


{'source_row_id': 'D-001', 'sale_id': 'S0001', 'sale_date': '2026-01-01', 'store_id': ' st01 ', 'product_id': 'P001', 'customer_id': 'C001', 'quantity': '1', 'unit_price': '80.00', 'discount_pct': '0', 'payment_method': 'UPI', 'record_version': '1'}
{'source_row_id': 'D-003', 'sale_id': 'S0003', 'sale_date': '01/01/2026', 'store_id': 'ST03', 'product_id': 'P003', 'customer_id': 'C003', 'quantity': '3', 'unit_price': '150.00', 'discount_pct': '10', 'payment_method': 'CASH', 'record_version': '1'}
{'source_row_id': 'D-004', 'sale_id': 'S0004', 'sale_date': '2026-01-01', 'store_id': 'ST04', 'product_id': 'P004', 'customer_id': 'C004', 'quantity': '1', 'unit_price': '', 'discount_pct': '0', 'payment_method': 'UPI', 'record_version': '1'}
{'source_row_id': 'D-005', 'sale_id': 'S0005', 'sale_date': '2026-01-01', 'store_id': 'ST05', 'product_id': 'P005', 'customer_id': 'C005', 'quantity': '2', 'unit_price': '120.00', 'discount_pct': '', 'payment_method': 'CARD', 'record_version': '1'}
{'sourc

## 03-02: Apply the documented cleaning rules

**Core concept Vibe:** Repair recoverable values; retain invalid records with a reason.

**Build Buddy action:** Use the visible clean_sales function and inspect each disposition.

**Expected checkpoint:** 68 input = 60 unique accepted + 2 duplicate drops + 6 rejected; 6 accepted rows repaired.


In [0]:
accepted, rejected, duplicate_ids, repaired_ids = clean_sales(dirty_rows)
assert (len(accepted), len(rejected), len(duplicate_ids), len(repaired_ids)) == (60, 6, 2, 6)
assert duplicate_ids == ['D-061', 'D-062']
assert rejected == read_rows('checks/expected/quarantine.csv')
for row in rejected:
    print(row['source_row_id'], row['rejection_reason'])
print('Repaired:', repaired_ids)
print('Duplicates:', duplicate_ids)


D-063 QUANTITY_INVALID
D-064 PRODUCT_UNKNOWN
D-065 STORE_UNKNOWN
D-066 DATE_INVALID
D-067 DISCOUNT_INVALID
D-068 CUSTOMER_UNKNOWN
Repaired: ['D-001', 'D-002', 'D-003', 'D-004', 'D-005', 'D-006']
Duplicates: ['D-061', 'D-062']


## 03-03: Prove the two initial routes agree

**Core concept Vibe:** Provenance can differ while business values agree.

**Build Buddy action:** Compare accepted rows with the canonical clean initial file, excluding source_row_id.

**Expected checkpoint:** All 60 business records match. The dirty route retains D-* source IDs.


In [0]:
assert business_records(accepted) == business_records(read_rows('data/sales_batch_01.csv'))
current = current_versions(accepted)
silver = sales_frame(current)
assert silver.count() == 60
assert_expected(silver, 'initial_sales.csv', ignore_provenance=True)


PASS: initial_sales.csv


## 03-04: Explain one fix and one rejection

**Core concept Vibe:** A trusted pipeline can explain its decisions.

**Build Buddy action:** Inspect D-004 and D-063 and compare a duplicate pair.

**Expected checkpoint:** D-004 price filled from P004; D-063 quarantined; normalized D-001 equals D-061. Day 1 can stop here.


In [0]:
fixed = next(row for row in accepted if row['source_row_id'] == 'D-004')
invalid = next(row for row in rejected if row['source_row_id'] == 'D-063')
assert fixed['unit_price'] == '40.00'
assert invalid['rejection_reason'] == 'QUANTITY_INVALID'
assert business_payload(next(r for r in accepted if r['source_row_id'] == 'D-001')) == business_payload(next(r for r in dirty_rows if r['source_row_id'] == 'D-061'))
print('D-004 price:', fixed['unit_price'])
print('D-063:', invalid['rejection_reason'])


D-004 price: 40.00
D-063: QUANTITY_INVALID


## 03-05: Persist the course layers

**Core concept Vibe:** Bronze retains raw strings; Silver holds valid current records.

**Build Buddy action:** Write dedicated training tables. Explicitly opt in to resetting an existing changed Silver table.

**Expected checkpoint:** 68 Bronze rows, 6 quarantined rows, 2 separately recorded duplicate rows, and 60 Silver sales.


In [0]:
RESET_INITIAL_STATE = False  # Set True only when deliberately restarting this exercise.
target = table_name('silver_sales')
if spark.catalog.tableExists(target):
    existing = collected_sales(spark.table(target))
    assert business_records(existing) == business_records(current) or RESET_INITIAL_STATE, (
        'Silver already contains different/later records. Keep the result, or deliberately set RESET_INITIAL_STATE=True to restart.'
    )
bronze = raw_frame('data/sales_dirty.csv').withColumn('source_file', F.lit('sales_dirty.csv')).withColumn('arrival_batch', F.lit('01'))
bronze.write.format('delta').mode('overwrite').saveAsTable(table_name('bronze_initial'))
quarantine_schema = ', '.join(f'{name} string' for name in list(rejected[0]))
quarantine = spark.createDataFrame([tuple(row[name] for name in rejected[0]) for row in rejected], quarantine_schema)
quarantine.write.format('delta').mode('overwrite').saveAsTable(table_name('quarantine'))
duplicates = raw_frame('data/sales_dirty.csv').where(F.col('source_row_id').isin(duplicate_ids))
duplicates.write.format('delta').mode('overwrite').saveAsTable(table_name('duplicate_rows'))
silver.write.format('delta').mode('overwrite').saveAsTable(target)
assert spark.table(target).count() == 60
assert spark.table(table_name('bronze_initial')).count() == 68
assert spark.table(table_name('quarantine')).count() == 6
print('PASS: initial course tables written.')


root
 |-- source_row_id: string (nullable = true)
 |-- sale_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 |-- discount_pct: decimal(5,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- record_version: integer (nullable = true)



## Restarting and teaching snapshots

This fixture overwrites its own per-file Bronze snapshot on rerun so physical row counts stay easy to explain. A production append-only Bronze design would retain ingestion events and separate file-processing metadata. This notebook does not delete source files or drop tables. After Day 2, leave RESET_INITIAL_STATE=False to protect the 80-sale result.

If you intentionally reset Silver, the Gold tables remain snapshots from their prior stage until notebooks 04 and 06 rebuild them. Their names include initial/final to make that distinction visible.
